In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install pyspark

In [3]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

# Create and config Spark
config = SparkConf().setAppName("bai01").setMaster("local[*]")
sparkContext = SparkContext.getOrCreate(conf=config)
sparkSession = SparkSession.builder.appName("bai01").getOrCreate()

In [9]:
data_path = "/content/drive/MyDrive/data/"

# Read file
movies_data = sparkContext.textFile(data_path + "movies.txt")

ratings1_data = sparkContext.textFile(data_path + "ratings_1.txt")
ratings2_data = sparkContext.textFile(data_path + "ratings_2.txt")

In [11]:
def splition_movie(line):
    parts = line.split(',', 2)  # Split text into MovieID, Title, Genres
    movie_id = int(parts[0])
    title = parts[1]
    return (movie_id, title)

movies_map = movies_data.map(splition_movie)
movies_dict = movies_map.collectAsMap()

In [13]:
# Split ratings: UserID, MovieID, Rating, Timestamp
def splition_rating(line):
    parts = line.split(',')
    movie_id = int(parts[1])
    rating = float(parts[2])
    return (movie_id, rating)

ratings1_map = ratings1_data.map(splition_rating)
ratings2_map = ratings2_data.map(splition_rating)


# Combine 2 file ratings
total_ratings = ratings1_map.union(ratings2_map)

In [14]:
# Convert (movie_id, rating) -> (movie_id, (rating, 1))
ratings_and_count = total_ratings.map(lambda x: (x[0], (x[1], 1)))

# Reduce key to calculate sum rate and count rating
movie_analysis = ratings_and_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))


In [15]:
# Tính điểm trung bình và thêm tên phim
# (movie_id, (average_rating, total_ratings, movie_title))
def calculation_average(row):
    movie_id, (sum_ratings, count) = row
    average_point = sum_ratings / count
    movie_title = movies_dict.get(movie_id, "No name Movie")
    return (movie_id, (average_point, count, movie_title))

movie_results = movie_analysis.map(calculation_average)

for movie in movie_results.collect():
    movie_id, (avg_rating, count, title) = movie
    print(f"{title} AverageRating: {avg_rating:.2f} (TotalRatings: {count})")

# Filter film which has >=5 (number of rating)
movies_filter_ratings = movie_results.filter(lambda x: x[1][1] >= 5)


if movies_filter_ratings.isEmpty():
    print("No movie found with at least 5 ratings.")
else:
    highest_rated_movie = movies_filter_ratings.max(key=lambda x: x[1][0])

    movie_id, (avg_rating, count, title) = highest_rated_movie
    print(f"\n{title} is the highest rated movie with an average rating of {avg_rating:.2f} among movies with at least 5 ratings.")


E.T. the Extra-Terrestrial (1982) AverageRating: 3.67 (TotalRatings: 18)
Psycho (1960) AverageRating: 4.00 (TotalRatings: 2)
Gladiator (2000) AverageRating: 3.61 (TotalRatings: 18)
Fight Club (1999) AverageRating: 3.50 (TotalRatings: 7)
The Lord of the Rings: The Fellowship of the Ring (2001) AverageRating: 3.89 (TotalRatings: 18)
The Terminator (1984) AverageRating: 4.06 (TotalRatings: 18)
The Godfather: Part II (1974) AverageRating: 4.00 (TotalRatings: 17)
The Silence of the Lambs (1991) AverageRating: 3.14 (TotalRatings: 7)
Mad Max: Fury Road (2015) AverageRating: 3.47 (TotalRatings: 18)
Lawrence of Arabia (1962) AverageRating: 3.44 (TotalRatings: 18)
Sunset Boulevard (1950) AverageRating: 4.36 (TotalRatings: 7)
The Social Network (2010) AverageRating: 3.86 (TotalRatings: 7)
No Country for Old Men (2007) AverageRating: 3.89 (TotalRatings: 18)
The Lord of the Rings: The Return of the King (2003) AverageRating: 3.82 (TotalRatings: 11)

Sunset Boulevard (1950) is the highest rated movi